# Knowledge Distillation

This notebook demonstrates how to distill knowledge from a large Teacher model into a smaller Student model.

**Use Case:** Improving the performance of a small, efficient model (e.g., CLIP ViT-Base) by learning from a larger, more powerful model (e.g., CLIP ViT-Large).


## Setup & Installation

In [1]:
# Clone the repository (if not running locally)
import os

# If running in Colab/Kaggle, uncomment below:
# !git clone https://github.com/fangzhensheng/vembed-factory.git
# os.chdir("vembed-factory")

print(f"Current working directory: {os.getcwd()}")

In [ ]:
# Install dependencies (using uv is recommended, but pip works too)
# !pip install -e .
!pip install -U pip
!pip install -r requirements.txt

## Data Preparation
We will prepare the Flickr30k dataset (or dummy data if Flickr30k is not available).

**Note:** For real training, you should download the Flickr30k images and place them in `data/flickr30k/images`.

In [ ]:
# This script generates dummy jsonl files if real data is missing
!python examples/prepare_data.py flickr30k

In [ ]:
# Verify Data
import os

if os.path.exists("data/flickr30k/train.jsonl"):
    print("Flickr30k data ready")
    print("Train:", "data/flickr30k/train.jsonl")
else:
    print("Using dummy data")
    print("Train:", "data/dummy/train.jsonl")

In [ ]:
# Config Data Paths
if os.path.exists("data/flickr30k/train.jsonl"):
    DATA_PATH = "data/flickr30k/train.jsonl"
    IMAGE_ROOT = "data/flickr30k"
    VAL_DATA_PATH = "data/flickr30k/val.jsonl"
else:
    DATA_PATH = "data/dummy/train.jsonl"
    IMAGE_ROOT = "data/dummy"
    VAL_DATA_PATH = ""

## Configuration

- **Teacher**: `openai/clip-vit-large-patch14` (Frozen)
- **Student**: `openai/clip-vit-base-patch32` (Trainable)
- **Method**: Relation Distillation (KL Divergence on Similarity Matrices)
- **Alpha**: 0.5 (50% Task Loss, 50% Distillation Loss)

In [ ]:
TEACHER_MODEL = "openai/clip-vit-large-patch14"
STUDENT_MODEL = "openai/clip-vit-base-patch32"

print(f"Teacher: {TEACHER_MODEL}")
print(f"Student: {STUDENT_MODEL}")

## Start Distillation Training

In [ ]:
!python run.py examples/clip_train.yaml \
    --data_path $DATA_PATH \
    --val_data_path "$VAL_DATA_PATH" \
    --image_root "$IMAGE_ROOT" \
    --config_override \
        output_dir=output_distill \
        epochs=3 \
        batch_size=32 \
        use_mrl=true \
        model_name="$STUDENT_MODEL" \
        teacher_model_name="$TEACHER_MODEL" \
        distillation_alpha=0.5 \
        distillation_temperature=2.0 \
        distillation_loss_type="kl"

## Evaluation Comparison
Compare the distilled model against the standard fine-tuned model (if available).

In [ ]:
if os.path.exists(VAL_DATA_PATH):
    print("Evaluating Distilled Model...")
    !python scripts/evaluate_simple.py \
        --model_path output_distill/checkpoint-epoch-3 \
        --data_path $VAL_DATA_PATH \
        --image_root $IMAGE_ROOT \
        --output_dir eval_results_distill

    !cat eval_results_distill/evaluation_report.md